# 1. What MANO computes

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

MANO is a function. You give it the shape and pose of a hand. It returns a mesh, and joints computed from that mesh by a formula.

```text
(global_orient, hand_pose, betas, transl)  →  778 vertices, 16 joints
```

It does not find a hand in a picture. An image, a depth map, or a sensor reading has to become the four inputs before MANO starts. Fitting, in chapter 14, is the search for a set of those four inputs whose output points lie close to points you have already measured.


## Four inputs

| Name | Length | What it decides |
|---|---|---|
| `global_orient` | 3 | How the whole hand rotates. These 3 numbers are the wrist axis-angle. |
| `hand_pose` | 45, or fewer PCA coefficients | How each of the 15 finger joints rotates. |
| `betas` | usually 10 | Whose hand this is: larger or smaller overall. It does not decide fist versus open palm. |
| `transl` | 3 | Where the hand sits. Metres. |

`global_orient` followed by `hand_pose` is 48 numbers, the full pose. Joint `i` uses `[3i : 3i+3]`. Joint 0 is the wrist, so `hand_pose` is exactly the 45 numbers left after dropping the first 3.

Run the next cell. It builds the four arrays and concatenates the 48-D pose.


In [2]:
import numpy as np

global_orient = np.zeros(3)
hand_pose = np.zeros(45)
betas = np.zeros(10)
transl = np.zeros(3)
full_pose = np.concatenate([global_orient, hand_pose])

print("global_orient", global_orient.shape)
print("hand_pose    ", hand_pose.shape)
print("betas        ", betas.shape)
print("transl       ", transl.shape)
print("full_pose    ", full_pose.shape)

joint = 2  # index PIP; the next chapter memorizes this table
print(f"joint {joint} occupies full_pose[{3 * joint}:{3 * joint + 3}]")
assert full_pose.shape == (48,)


global_orient (3,)
hand_pose     (45,)
betas         (10,)
transl        (3,)
full_pose     (48,)
joint 2 occupies full_pose[6:9]


## Two outputs

Vertices are 778 rows by 3 columns. Each row is a point `(x, y, z)` in metres. There are 1538 triangles. Each triangle is three vertex indices. The faces live in the model file and do not change with pose.

The 16 joints are also 16 points in 3D. There is no separate skeleton file. Shape edits the vertices first, joints are a linear combination of those vertices, and pose then carries the joints along. Changing `betas` therefore moves the rest positions of the joints.

The five fingertips are not among these 16 joints. They are vertices with fixed indices. The usual "21 points" are the 16 joints plus the 5 fingertip vertices. Chapter 3 lists the indices. Chapter 10 reads them off the official mesh.

## Units, and zero

Angles inside the model are radians. Forty degrees is `40 * π / 180`.


In [3]:
degrees = 40.0
radians = np.deg2rad(degrees)
print(f"{degrees} degrees = {radians:.6f} radians")
print("40 * pi / 180 =", 40 * np.pi / 180)
assert np.isclose(radians, 0.6981317)


40.0 degrees = 0.698132 radians
40 * pi / 180 = 0.6981317007977318


Zero has two meanings. Chapter 11 separates them. The default to remember now is: `flat_hand=True` and a pose of all zeros is the registered flat hand, called the template. `betas` of all zeros is the mean body shape of the training scans. `transl` of all zeros adds no translation.

## The order to keep in mind

The next chapters take the computation apart. The order is always:

1. Edit the template vertices with `betas`.
2. Regress 16 rest joints from the edited vertices.
3. Add a flesh correction built from the finger rotations.
4. Move the joints along the parent tree.
5. Let every vertex follow the bones named by its skinning weights, then add `transl`.

You do not need to write those five steps yet. You need to be able to say: shape chooses whose hand, pose chooses what that hand is doing, and translation chooses where the hand is.

The next chapter makes "three numbers are a rotation" automatic. The joint tree and the formula both sit on that.
